# Phase 5: Baseline Models\n\nSeasonal Naive + Gradient Descent Linear Regression (NumPy from scratch)

In [ ]:
import sys\nsys.path.append('..')\n\nimport pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport joblib\nfrom datetime import datetime\n\nfrom retail_iq.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, MODEL_DIR\nfrom retail_iq.preprocessing import load_raw_data, preprocess_dates, merge_datasets, detect_outliers_iqr\nfrom retail_iq.features import FastFeatureEngineer\nfrom retail_iq.models import GDLinearRegressor, SeasonalNaive, ModelPersistence\nfrom retail_iq.evaluation import evaluate_model, create_metrics_table, check_residual_bias

## 1. Load and Prepare Data

In [ ]:
# Load raw data\ntrain, test, stores, oil, holidays, transactions = load_raw_data()\n\n# Preprocess dates\ntrain, test, oil, holidays, transactions = preprocess_dates([train, test, oil, holidays, transactions])\n\n# Merge\ndf = merge_datasets(train, stores, oil, holidays, transactions)\nprint(f"Merged shape: {df.shape}")

In [ ]:
# Temporal holdout: Train < 2017-08-16, Test = Aug 16-31\nCUTOFF = pd.Timestamp('2017-08-16')\n\ntrain_df = df[df['date'] < CUTOFF].copy()\ntest_df = df[(df['date'] >= CUTOFF) & (df['date'] < '2017-09-01')].copy()\n\nprint(f"Train: {len(train_df)} rows, {train_df['date'].min()} to {train_df['date'].max()}")\nprint(f"Test: {len(test_df)} rows, {test_df['date'].min()} to {test_df['date'].max()}")

## 2. Feature Engineering (FastFeatureEngineer)

In [ ]:
# Build features for train and test\ndef build_features(df, transactions, oil, stores, holidays):\n    fe = FastFeatureEngineer(df, transactions=transactions, oil_price=oil, store_meta=stores, holidays=holidays)\n    fe.add_temporal_features()\n    fe.add_lag_and_rolling(lags=[1,7,14,365], windows=[7,14,28])\n    fe.add_onpromotion_features()\n    fe.add_macroeconomic_features()\n    fe.add_transaction_features()\n    fe.add_store_metadata()\n    return fe.transform()\n\ntrain_feat = build_features(train_df, transactions, oil, stores, holidays)\ntest_feat = build_features(test_df, transactions, oil, stores, holidays)\n\n# Drop rows with NaN from 365-day lag\ntrain_feat = train_feat.dropna(subset=['sales_lag_365d'])\ntest_feat = test_feat.dropna(subset=['sales_lag_365d'])\n\nprint(f"Train features: {train_feat.shape}")\nprint(f"Test features: {test_feat.shape}")

## 3. Baseline 1: Seasonal Naive

In [ ]:
# Seasonal Naive: predict sales_lag_365d\ny_test = test_feat['sales'].values\ny_pred_naive = test_feat['sales_lag_365d'].fillna(0).values\n\nnaive_metrics = evaluate_model(y_test, y_pred_naive, "SeasonalNaive")\nprint(naive_metrics)

In [ ]:
# Save naive predictions\nnaive_df = pd.DataFrame({'actual': y_test, 'predicted': y_pred_naive})\nnaive_df.to_csv(f'{MODEL_DIR}/naive_predictions.csv', index=False)

## 4. Baseline 2: Gradient Descent Linear Regression

In [ ]:
# Select numeric features for linear model\nexclude_cols = ['sales', 'date', 'family', 'city', 'state', 'holiday_type', 'locale', 'locale_name', 'description', 'transferred']\nfeature_cols = [c for c in train_feat.columns if c not in exclude_cols]\n\n# Fill any remaining NaNs\nX_train = train_feat[feature_cols].fillna(0).values\ny_train = train_feat['sales'].values\nX_test = test_feat[feature_cols].fillna(0).values\ny_test = test_feat['sales'].values\n\nprint(f"Features: {len(feature_cols)}")\nprint(f"X_train: {X_train.shape}, y_train: {y_train.shape}")

In [ ]:
# Train GD Linear Regression\ngd_model = GDLinearRegressor(\n    learning_rate=0.01,\n    iterations=1000,\n    l1_penalty=0.001,\n    l2_penalty=0.01,\n    random_state=42\n)\n\ngd_model.fit(X_train, y_train)

In [ ]:
# Plot convergence\nplt.figure(figsize=(10, 4))\nplt.plot(gd_model.loss_history)\nplt.xlabel('Iteration')\nplt.ylabel('MSE (log scale)')\nplt.title('GD Linear Regression Convergence')\nplt.yscale('log')\nplt.savefig(f'{MODEL_DIR}/gd_convergence.png', dpi=150, bbox_inches='tight')\nplt.show()

In [ ]:
# Predict and evaluate\ny_pred_gd = gd_model.predict(X_test)\ngd_metrics = evaluate_model(y_test, y_pred_gd, "GD_Linear")\nprint(gd_metrics)

In [ ]:
# Check for bias\nhas_bias, bias_pct = check_residual_bias(y_test, y_pred_gd, threshold_pct=5.0)\nprint(f"Mean residual: {bias_pct:.2f}% of mean actual")\nprint(f"Has systematic bias: {has_bias}")

In [ ]:
# Save model\nmodel_path = f'{MODEL_DIR}/gd_linear_{datetime.now().strftime("%Y%m%d")}_v1.pkl'\nparams_path = f'{MODEL_DIR}/gd_linear_params.json'\n\nModelPersistence.save_model(gd_model, model_path, params_path)\nprint(f"Saved to {model_path}")

In [ ]:
# Verify reload\nverification = ModelPersistence.verify_reload(gd_model, X_test, model_path)\nprint(f"Reload verification (bit-for-bit): {verification}")

## 5. Model Comparison

In [ ]:
# Comparison table\nresults = [naive_metrics, gd_metrics]\ncomparison_df = create_metrics_table(results)\nprint(comparison_df)\n\n# Save\ncomparison_df.to_csv(f'{MODEL_DIR}/baseline_metrics.csv', index=False)

In [ ]:
# Actual vs Predicted plot (sample)\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\nsample_idx = np.random.choice(len(y_test), 1000, replace=False)\n\naxes[0].scatter(y_test[sample_idx], y_pred_naive[sample_idx], alpha=0.3)\naxes[0].plot([0, y_test.max()], [0, y_test.max()], 'r--')\naxes[0].set_xlabel('Actual Sales')\naxes[0].set_ylabel('Predicted Sales')\naxes[0].set_title('Seasonal Naive')\n\naxes[1].scatter(y_test[sample_idx], y_pred_gd[sample_idx], alpha=0.3)\naxes[1].plot([0, y_test.max()], [0, y_test.max()], 'r--')\naxes[1].set_xlabel('Actual Sales')\naxes[1].set_ylabel('Predicted Sales')\naxes[1].set_title('GD Linear Regression')\n\nplt.tight_layout()\nplt.savefig(f'{MODEL_DIR}/baseline_actual_vs_pred.png', dpi=150, bbox_inches='tight')\nplt.show()

## Summary\n\n- Seasonal Naive: Zero-training baseline using 365-day lag\n- GD Linear: NumPy-only implementation with L1+L2 regularization\n- Both models saved with verification\n\nNext: Phase 6 - Advanced Models (XGBoost, LightGBM + Optuna tuning)